# Ablation B — openFDA RAG on E7 Inference

**Hypothesis:** Compact distilled students lack factual capacity but can be augmented
with retrieval at inference. We use openFDA drug labels (public domain, free, no auth)
as the knowledge source.

**Pipeline per question:**
1. Extract drug names from question (regex against prefetched openFDA drug name list)
2. Query openFDA `/drug/label.json` for each drug — fetch warnings, contraindications, drug_interactions sections
3. Concatenate top snippets as a CONTEXT block prepended to the prompt
4. Run E7 student inference

**Method:** E7 (best single-objective baseline)
**Sizes:** 1.5B + 3B + 7B
**Samples:** 50 (subset of test set, same 50 used in MLP ablation for paired comparison)
**Output:** `data/e7_rag_inference_50_TESTONLY.json`

In [1]:
# Cell 0: Config + load eval questions
import os, sys, json, re, time, urllib.parse
import urllib.request
import torch

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"
# PROJECT_DIR = os.path.expanduser("~/kd_project")

DATA_DIR = os.path.join(PROJECT_DIR, "data")
RUNS_DIR = os.path.join(PROJECT_DIR, "runs")

N_EVAL = 100
GEN_KW = dict(max_new_tokens=2000, do_sample=False)

# Use first 50 of the 100-question test set (deterministic — paired comparison with MLP ablation)
QUESTIONS_FILE = os.path.join(DATA_DIR, "eval_questions_100.json")
with open(QUESTIONS_FILE) as f:
    q_data = json.load(f)
eval_questions = q_data["questions"][:N_EVAL]
print(f"Loaded {len(eval_questions)} questions")

# E7 adapter paths
E7_ADAPTERS = {
    "qwen25_1p5b_base": os.path.join(DATA_DIR, "e7_decision_only_sft", "qwen25_1p5b_base"),
    "qwen25_3b_base":   os.path.join(DATA_DIR, "e7_decision_only_sft", "qwen25_3b_base"),
    "qwen25_7b_base":   os.path.join(DATA_DIR, "e7_decision_only_sft", "qwen25_7b_base"),
}

STUDENT_PATHS = {
    "qwen25_1p5b_base": "Qwen/Qwen2.5-1.5B",
    "qwen25_3b_base":   "Qwen/Qwen2.5-3B",
    "qwen25_7b_base":   "Qwen/Qwen2.5-7B",
}

OUT_FILE = os.path.join(DATA_DIR, f"e7_rag_inference_{N_EVAL}_TESTONLY.json")

print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'NONE'}")
print(f"Output: {OUT_FILE}")

Loaded 100 questions
GPU: NVIDIA GeForce RTX 4090
Output: C:\Users\adishalit1\Desktop\kd_project\data\e7_rag_inference_100_TESTONLY.json


In [2]:
# Cell 1 (REPLACEMENT): openFDA RAG with prefetched drug dictionary
# Fully local at inference time. One-time fetch of FDA drug names from openFDA.

import re, urllib.request, urllib.parse, json, time

RAG_CACHE = os.path.join(DATA_DIR, "openfda_cache")
os.makedirs(RAG_CACHE, exist_ok=True)

OPENFDA_BASE = "https://api.fda.gov/drug/label.json"
DRUG_DICT_FILE = os.path.join(DATA_DIR, "openfda_drug_dictionary.json")

def http_get(url, timeout=15):
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "research/1.0"})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except Exception:
            if attempt == 2:
                return None
            time.sleep(1 + attempt)
    return None

# ── ONE-TIME: Fetch drug name dictionary from openFDA ──
def build_drug_dictionary(force=False):
    """Fetch top generic + brand names from openFDA. Cached locally."""
    if not force and os.path.exists(DRUG_DICT_FILE):
        with open(DRUG_DICT_FILE) as f:
            return json.load(f)
    
    print("Building drug dictionary from openFDA (one-time)...")
    drug_set = set()
    
    # Fetch top N generic names by count
    for field in ["openfda.generic_name.exact", "openfda.brand_name.exact",
                  "openfda.substance_name.exact"]:
        url = f"{OPENFDA_BASE}?count={field}&limit=1000"
        data = http_get(url)
        if data and "results" in data:
            for entry in data["results"]:
                term = entry.get("term", "").strip().lower()
                # Filter: must be 5+ chars, no whitespace (single word), alphabetic
                if len(term) >= 5 and " " not in term and term.replace("-","").isalpha():
                    drug_set.add(term)
            print(f"  {field}: {len(drug_set)} unique drugs total")
    
    # Save dictionary
    drug_list = sorted(drug_set)
    with open(DRUG_DICT_FILE, "w") as f:
        json.dump(drug_list, f)
    print(f"  ✅ Saved {len(drug_list)} drugs → {DRUG_DICT_FILE}")
    return drug_list

DRUG_DICT = set(build_drug_dictionary())
print(f"\nDrug dictionary loaded: {len(DRUG_DICT)} drugs")
print(f"Sample: {sorted(DRUG_DICT)[:10]}")

# ── LOCAL: Extract drug candidates by dictionary lookup ──
def extract_drug_candidates(text):
    """Find drug names in text by matching against the FDA dictionary.
    
    Fully local — no API calls at inference time.
    Returns up to 6 unique drug names found in the question.
    """
    text_lower = text.lower()
    found = []
    seen = set()
    
    # Tokenize: split on non-alpha boundaries to get word candidates
    tokens = re.findall(r"\b[a-z]{5,}\b", text_lower)
    
    for tok in tokens:
        if tok in DRUG_DICT and tok not in seen:
            seen.add(tok)
            found.append(tok)
            if len(found) >= 6:
                break
    
    return found

# ── openFDA label fetching (cached) ──
def fetch_drug_label(drug_name):
    """Query openFDA for a drug, return relevant label sections. Cached."""
    safe = re.sub(r"[^a-z0-9]+", "_", drug_name.lower())
    cache_file = os.path.join(RAG_CACHE, f"{safe}.json")
    
    if os.path.exists(cache_file):
        try:
            with open(cache_file) as f:
                return json.load(f)
        except: pass
    
    queries = [
        f'openfda.generic_name:"{drug_name.lower()}"',
        f'openfda.brand_name:"{drug_name.lower()}"',
    ]
    
    result = None
    for q in queries:
        url = f"{OPENFDA_BASE}?search={urllib.parse.quote(q)}&limit=1"
        data = http_get(url)
        if data and "results" in data and data["results"]:
            r = data["results"][0]
            result = {
                "drug": drug_name,
                "warnings": " ".join(r.get("warnings", []) or r.get("warnings_and_cautions", []))[:800],
                "contraindications": " ".join(r.get("contraindications", []))[:600],
                "drug_interactions": " ".join(r.get("drug_interactions", []))[:800],
                "dosage": " ".join(r.get("dosage_and_administration", []))[:400],
            }
            break
    
    if result is None:
        result = {"drug": drug_name, "warnings": "", "contraindications": "",
                  "drug_interactions": "", "dosage": ""}
    
    with open(cache_file, "w") as f:
        json.dump(result, f)
    return result

def build_rag_context(question, max_drugs=4):
    """Build a RAG context block to prepend to a question. Fully local."""
    candidates = extract_drug_candidates(question)
    if not candidates:
        return None, []
    
    snippets = []
    used_drugs = []
    for drug in candidates[:max_drugs]:
        label = fetch_drug_label(drug)
        if any([label["contraindications"], label["drug_interactions"], label["warnings"]]):
            used_drugs.append(drug)
            parts = [f"### {drug}"]
            if label["contraindications"]:
                parts.append(f"Contraindications: {label['contraindications']}")
            if label["drug_interactions"]:
                parts.append(f"Drug interactions: {label['drug_interactions']}")
            if label["warnings"]:
                parts.append(f"Warnings: {label['warnings']}")
            snippets.append("\n".join(parts))
    
    if not snippets:
        return None, []
    
    context = "RELEVANT CLINICAL CONTEXT FROM FDA DRUG LABELS:\n\n" + "\n\n".join(snippets) + "\n\n---\n\n"
    return context, used_drugs

# Quick test
print("\n" + "="*60)
print("Testing local drug extraction...")
print("="*60)
test_q = eval_questions[0]["prompt"]
print(f"Question (first 200 chars): {test_q[:200]}")
print()
ctx, drugs = build_rag_context(test_q)
print(f"Drugs found: {drugs}")
if ctx:
    print(f"Context length: {len(ctx)} chars")
    print(f"Preview: {ctx[:400]}...")
else:
    print("No context retrieved")


Drug dictionary loaded: 486 drugs
Sample: ['abacavir', 'abrotanum', 'absinthium', 'acetaminophen', 'acetazolamide', 'acetylcysteine', 'acyclovir', 'adamas', 'adapalene', 'adenosine']

Testing local drug extraction...
Question (first 200 chars): QUESTION:
You are a clinical pharmacology decision assistant.
Always base your answers on verified medical knowledge, and avoid speculation.

Patient profile:
- Age: 60
- Sex: Female
- Weight: 73 kg
-

Drugs found: ['lisinopril', 'furosemide', 'hydrochlorothiazide']
Context length: 6410 chars
Preview: RELEVANT CLINICAL CONTEXT FROM FDA DRUG LABELS:

### lisinopril
Contraindications: CONTRAINDICATIONS Lisinopril and hydrochlorothiazide tablets are contraindicated in patients who are hypersensitive to any component of this product and in patients with a history of angioedema related to previous treatment with an angiotensin converting enzyme inhibitor and in patients with hereditary or idiopathic...


In [3]:
# Cell 2: Pre-fetch all RAG contexts (cache the openFDA queries)
print("Pre-fetching RAG contexts for all 50 questions...")
print("(One-time cost — subsequent runs hit local cache)")

contexts = {}
drugs_found = {}
for i, q in enumerate(eval_questions):
    ctx, drugs = build_rag_context(q["prompt"])
    contexts[q["id"]] = ctx
    drugs_found[q["id"]] = drugs
    if (i+1) % 10 == 0:
        n_with = sum(1 for c in contexts.values() if c)
        print(f"  {i+1}/{len(eval_questions)} | {n_with} with context")

n_with_ctx = sum(1 for c in contexts.values() if c)
total_drugs = sum(len(d) for d in drugs_found.values())
print(f"\n✅ {n_with_ctx}/{len(eval_questions)} questions have RAG context")
print(f"✅ {total_drugs} drug labels retrieved total")

# Show drug coverage stats
from collections import Counter
all_drugs = [d for ds in drugs_found.values() for d in ds]
print(f"\nMost common drugs retrieved:")
for drug, count in Counter(all_drugs).most_common(15):
    print(f"  {drug}: {count}")

Pre-fetching RAG contexts for all 50 questions...
(One-time cost — subsequent runs hit local cache)
  10/100 | 9 with context
  20/100 | 18 with context
  30/100 | 28 with context
  40/100 | 38 with context
  50/100 | 46 with context
  60/100 | 55 with context
  70/100 | 62 with context
  80/100 | 69 with context
  90/100 | 79 with context
  100/100 | 89 with context

✅ 89/100 questions have RAG context
✅ 179 drug labels retrieved total

Most common drugs retrieved:
  loratadine: 20
  warfarin: 13
  quetiapine: 10
  prednisone: 9
  valacyclovir: 9
  furosemide: 8
  metformin: 8
  amoxicillin: 8
  lisinopril: 7
  hydrochlorothiazide: 7
  fluconazole: 7
  naproxen: 6
  rifampin: 6
  sildenafil: 6
  spironolactone: 5


In [ ]:
# Cell 3: Run inference with RAG-augmented prompts
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import gc

# Resume support
if os.path.exists(OUT_FILE):
    with open(OUT_FILE) as f:
        result = json.load(f)
    done_models = set(result.get("models", {}).keys())
    print(f"Resume: {done_models} already done")
else:
    result = {
        "meta": {"source": "eval_questions_100.json[:50]", "n_samples": N_EVAL,
                 "method": "E7 + openFDA RAG", "rag_drugs_per_q": drugs_found},
        "models": {},
        "samples": [{"id": q["id"], "prompt": q["prompt"],
                     "rag_context": contexts.get(q["id"]),
                     "rag_drugs": drugs_found.get(q["id"], []),
                     "outputs": {}} for q in eval_questions],
    }
    done_models = set()

@torch.inference_mode()
def run_rag_inference(model, tokenizer):
    answers = []
    for sample in result["samples"]:
        prompt = sample["prompt"]
        rag_ctx = sample.get("rag_context")
        # Inject RAG context BEFORE the original prompt
        full_input = (rag_ctx + prompt) if rag_ctx else prompt
        inputs = tokenizer(full_input, return_tensors="pt", truncation=True, max_length=4096)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
        out = model.generate(**inputs, **GEN_KW)
        decoded = tokenizer.decode(out[0], skip_special_tokens=True)
        answer = decoded[len(full_input):].lstrip() if decoded.startswith(full_input) else decoded.strip()
        answers.append(answer)
    return answers

for sname, model_path in STUDENT_PATHS.items():
    if sname in done_models:
        print(f"⏩ {sname} already done")
        continue

    adapter = E7_ADAPTERS[sname]
    if not os.path.exists(os.path.join(adapter, "adapter_config.json")):
        print(f"⚠️ {sname}: E7 adapter not found at {adapter}")
        continue

    print(f"\n{'='*60}")
    print(f"  Loading {sname} + E7 adapter")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    if "7b" in sname:
        bnb = BitsAndBytesConfig(load_in_8bit=True)
        base = AutoModelForCausalLM.from_pretrained(
            model_path, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    else:
        base = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)

    model = PeftModel.from_pretrained(base, adapter)
    model.eval()

    t0 = time.time()
    answers = run_rag_inference(model, tokenizer)
    elapsed = time.time() - t0
    print(f"  ✅ {len(answers)} samples in {elapsed/60:.1f} min")

    result["models"][sname] = {"adapter": adapter, "path": model_path, "rag": True}
    for sample, answer in zip(result["samples"], answers):
        sample["outputs"][sname] = {"answer": answer}

    with open(OUT_FILE, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"  Saved → {OUT_FILE}")

    del model, base, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ All RAG inference done")

c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  Loading qwen25_1p5b_base + E7 adapter


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 154.38it/s]
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Settin

  ✅ 100 samples in 11.3 min
  Saved → C:\Users\adishalit1\Desktop\kd_project\data\e7_rag_inference_100_TESTONLY.json

  Loading qwen25_3b_base + E7 adapter


Loading weights: 100%|██████████| 434/434 [00:04<00:00, 108.27it/s]
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open

  ✅ 100 samples in 16.8 min
  Saved → C:\Users\adishalit1\Desktop\kd_project\data\e7_rag_inference_100_TESTONLY.json

  Loading qwen25_7b_base + E7 adapter


Loading weights: 100%|██████████| 339/339 [00:15<00:00, 22.49it/s]
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`

  ✅ 100 samples in 51.4 min
  Saved → C:\Users\adishalit1\Desktop\kd_project\data\e7_rag_inference_100_TESTONLY.json

✅ All RAG inference done


: 